# 96-Well Plate: Data Import (Pilot Study)

This notebook oads the raw plate-reader Excel files for the 96-well pilot experiment, reshapes them from wide (one column per well) into tidy long format (one row per measurement), attaches experimental metadata (which peptide, which wine, which replicate each well corresponds to), and saves the combined result as a single CSV for downstream analysis.

The experiment includes 6 wines (W1-W6), 14 peptides, a single fixed metal and dye (Cu2+-PCV). Because each plate reader could only fit so many peptide rows, this pilot's 14 peptides were measured across two separate plate files per wine pair (peptides 1-7, then peptides 8-14). This is why the plate list below has twice as many entries as wine pairs. Each plate also includes a wine-present/no-peptide control row.

The relationship between this import and the 384-well import include: the same load_plate, tidy_plate, and add_plate_map functions are reused unchanged between
this notebook and 01_384well_Import.ipynb. Only the plate map construction differs, because the two plate layouts are physically different (rows vs. dedicated control rows). This is the pilot dataset and 01_384well_Import.ipynb builds the primary dataset.

## Imports

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

## Loading functions

load_plate and tidy_plate are shared, unmodified, between the 96-well and 384-well pipelines. 

In [ ]:
#Loads one raw plate-reader Excel file and returns a wavelength x well absorbance table
#The header row isn't at a fixed position across files, so it's located by searchng for the "Wavel." label

def load_plate(file_path):

    #Read the first 100 rows to find the header
    preview = pd.read_excel(
        file_path,
        header = None,
        nrows = 100
    )

    #Find the row containing Wavel.
    matches = preview.eq("Wavel.")

    if not matches.any().any():
        raise ValueError(f"Could not find 'Wavel.' in {file_path}")

    header_row = matches.any(axis = 1).idxmax()

    #Read the data
    df = pd.read_excel(
        file_path,
        header = header_row,
        nrows = 226
    )

    #Rename the wavelength column
    df = df.rename(columns = {"Wavel.": "Wavelength"})

    #Ensure numeric
    df["Wavelength"] = pd.to_numeric(df["Wavelength"])

    return df

In [ ]:
#Reshapes one plate from wide (one column per well) to long (one row per wavelength x well measurement)
#Each row is tagged with which plate file it came from

def tidy_plate(df, plate_name):
   
    df_long = df.melt(
        id_vars = "Wavelength",
        var_name = "Well",
        value_name = "Absorbance"
    )

    #Keep track of which plate the data came from
    df_long["Plate"] = plate_name

    return df_long

## 96-well plate map

The 96-well plate has 8 rows (A-H) and 12 columns. Rows A-G are the 7 real peptide rows for this plate's peptide batch (peptide_start selects which 7 of the 14 total
peptides (1-7 or 8-14) this particular physical plate covers).

Row H is a HEPES-only control row, not an 8th peptide. This row is wine-present/no-peptide, it isolates the wine's own colour, independent of any sensor chemistry. Columns 1 and 12 are a second, different control: peptide-present/no-wine, isolating each peptide's own background chemistry.

In [ ]:
#Builds the well mapping for one 96-well plate (peptide, sensor, wine, replicate)
#Rows A-G are real peptide rows (peptide_start controls which 7 of the 14 total peptides this plate covers)
#Row H is HEPES-only, wine present/no peptide control row and is labelled "wine control"

def create_plate_map_96(peptide_start = 1, wine_names = ("W1", "W2")):

    rows = list("ABCDEFGH")
    columns = range(1, 13)

    plate_map = []

    for row_index, row in enumerate(rows):

        #Row H (index 7) is the HEPES-only wine-control row
        if row_index == 7:
            peptide = None
            sensor = "Wine control"
        else:
            peptide = peptide_start + row_index
            sensor = f"Cu-PCV-P{peptide}"

        for column in columns:
            well = f"{row}{column}"

            #Controls: HEPES buffer only, no wine 
            if column in (1, 12):
                sample = "HEPES"
                replicate = None

            #Wine 1
            elif 2 <= column <= 6:
                sample = wine_names[0]
                replicate = column - 1

            #Wine 2
            else:
                sample = wine_names[1]
                replicate = column - 6

            plate_map.append({
                "Well": well,
                "Peptide": peptide,
                "Sensor": sensor,
                "Wine": sample,
                "Replicate": replicate
            })

    return pd.DataFrame(plate_map)

In [ ]:
#Attach experimental metadata (peptide, wine, replicate) to a tidy plate dataframe by joining on well

def add_plate_map(df_long, plate_map):
   
    df = df_long.merge(
        plate_map,
        on = "Well",
        how = "left"
    )

    return df

In [ ]:
#Runs one raw plate file through the full load, tidy, map pipieline in one call

def process_plate(
            file_path,
            plate_name,
            peptide_start,
            wine_names
        ):

    plate = load_plate(file_path)

    plate_long = tidy_plate(plate, plate_name)

    plate_map = create_plate_map_96(peptide_start, wine_names)

    plate_complete = add_plate_map(plate_long, plate_map)

    return plate_complete

## Plate registry

One entry per plate file. Each wine pair (W1&2, W3&4, W5&6) was measured across two files, one for peptides 1-7 and one for peptides 8-14, reflecting the plate reader's size limit.

In [15]:
plates = [
    {
        "file_path": "../../data/raw/W1&2_pep1-7.xlsx",
        "plate_name": "W1_2_P1_7",
        "peptide_start": 1,
        "wine_names": ("W1", "W2")
    },
    {
        "file_path": "../../data/raw/updated_W1&2_peptides8-14.xlsx",
        "plate_name": "W1_2_P8_14",
        "peptide_start": 8,
        "wine_names": ("W1", "W2")
    },
    {
        "file_path": "../../data/raw/W3&4_pep1-7.xlsx",
        "plate_name": "W3_4_P1_7",
        "peptide_start": 1,
        "wine_names": ("W3", "W4")
    },
    {
        "file_path": "../../data/raw/updated_W3&4_peptides8-14.xlsx",
        "plate_name": "W3_4_P8_14",
        "peptide_start": 8,
        "wine_names": ("W3", "W4")
    },
    {
        "file_path": "../../data/raw/W5&6_pep1-7.xlsx",
        "plate_name": "W5_6_P1_7",
        "peptide_start": 1,
        "wine_names": ("W5", "W6")
    },
    {
        "file_path": "../../data/raw/W5&6_pep8-14.xlsx",
        "plate_name": "W5_6_P8_14",
        "peptide_start": 8,
        "wine_names": ("W5", "W6")
    }
]

## Run the pipeline

Each plate is processed independently and wrapped in a try/except so that one bad file doesn't stop the whole run. Any failure is printed and skipped, rather than silently dropped.

In [16]:
all_plates = []

for plate in plates:
    try:
        print(f"Processing {plate['plate_name']}...")
        df = process_plate(**plate)
        all_plates.append(df)
        print("Success")

    except Exception as e:
        print(f"Failed: {plate['plate_name']}")
        print(e)
        print()

master_df_96 = pd.concat(all_plates, ignore_index = True)

print("Done!")
print(master_df_96.shape)

Processing W1_2_P1_7...
Success
Processing W1_2_P8_14...
Success
Processing W3_4_P1_7...
Success
Processing W3_4_P8_14...
Success
Processing W5_6_P1_7...
Success
Processing W5_6_P8_14...
Success
Done!
(125817, 8)


## Quality check

Per-plate summary as a check before saving. `Peptides` should now read 7 per plate (not 8), confirming row H is excluded from peptide numbering.

In [18]:
def plate_summary(df):
    """One row per plate: counts and ranges used as a sanity check that
    each plate was parsed as expected before saving."""

    summary = (
        df
        .groupby("Plate")
        .agg(
            Measurements = ("Absorbance", "count"),
            Wells = ("Well", "nunique"),
            Wavelength_Min = ("Wavelength", "min"),
            Wavelength_Max = ("Wavelength", "max"),
            Peptides = ("Peptide", "nunique"),
            Sensors = ("Sensor", "nunique"),
            Wines = ("Wine", "nunique")
        )
    )

    return summary

plate_summary(master_df_96)

,Measurements,Wells,Wavelength_Min,Wavelength_Max,Peptides,Sensors,Wines
Plate,,,,,,,
W1_2_P1_7,21696,96,350,800,7,8,3
W1_2_P8_14,21696,96,350,800,7,8,3
W3_4_P1_7,21696,96,350,800,7,8,3
W3_4_P8_14,18693,93,350,750,7,8,3
W5_6_P1_7,20792,92,350,800,7,8,3
W5_6_P8_14,21244,94,300,750,7,8,3


## Save

Saved as master_df_96.csv.

In [19]:
master_df_96.to_csv(
    "../../data/processed/master_df_96.csv",
    index = False
)